In [1]:
import os, sys
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/wdi_solow_tidy.csv")

# sort is VERY important for time shifting
df = df.sort_values(["country", "year"]).reset_index(drop=True)

df.head()

# GDP per capita 10 years later
df["gdp_pc_t10"] = df.groupby("country")["gdp_per_capita"].shift(-1)

# 10-year average annual growth
df["growth_10y_avg"] = (
    np.log(df["gdp_pc_t10"]) - np.log(df["gdp_per_capita"])
) / 10

df[["country", "year", "gdp_per_capita", "gdp_pc_t10", "growth_10y_avg"]].head(10)

g = 0.02
delta = 0.03

# build n + g + δ
df["n_g_delta"] = df["population_growth"] / 100 + g + delta

# remove invalid values BEFORE log
df = df[
    (df["gdp_per_capita"] > 0) &
    (df["investment_rate"] > 0) &
    (df["n_g_delta"] > 0)
].copy()

# log features
df["ln_y0"] = np.log(df["gdp_per_capita"])
df["ln_s"] = np.log(df["investment_rate"] / 100)
df["ln_ngd"] = np.log(df["n_g_delta"])

ml_df = df.dropna(subset=[
    "growth_10y_avg",
    "ln_y0",
    "ln_s",
    "ln_ngd"
]).copy()

ml_df[["country", "year", "growth_10y_avg", "ln_y0", "ln_s", "ln_ngd"]].head()

features_theory = ["ln_y0", "ln_s", "ln_ngd"]
target = "growth_10y_avg"

print("Dataset shape:", ml_df.shape)
print("\nYears distribution:")
print(ml_df["year"].value_counts().sort_index())

ml_df.groupby("country")["year"].count().describe()

train_df = ml_df[ml_df["year"].isin([1990, 2000])].copy()
test_df  = ml_df[ml_df["year"] == 2010].copy()

print("Train years:")
print(train_df["year"].value_counts().sort_index())

print("\nTest years:")
print(test_df["year"].value_counts().sort_index())

features = ["ln_y0", "ln_s", "ln_ngd"]

X_train = train_df[features]
y_train = train_df["growth_10y_avg"]

X_test = test_df[features]
y_test = test_df["growth_10y_avg"]

print(X_train.shape, X_test.shape)

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.dummy import DummyRegressor
import numpy as np

def evaluate(model, X_train, y_train, X_test, y_test, name):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    return {
        "model": name,
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(y_test, preds),
        "R2_oos": r2_score(y_test, preds)
    }

results = []

# Mean baseline
results.append(evaluate(
    DummyRegressor(strategy="mean"),
    X_train, y_train, X_test, y_test,
    "Mean"
))

# Linear regression
results.append(evaluate(
    LinearRegression(),
    X_train, y_train, X_test, y_test,
    "Linear"
))

# Ridge
results.append(evaluate(
    Ridge(alpha=1.0),
    X_train, y_train, X_test, y_test,
    "Ridge"
))

# Lasso
results.append(evaluate(
    Lasso(alpha=0.001),
    X_train, y_train, X_test, y_test,
    "Lasso"
))

import pandas as pd
results_df = pd.DataFrame(results).sort_values("RMSE")

results_df

Dataset shape: (571, 11)

Years distribution:
year
1990    166
2000    196
2010    209
Name: count, dtype: int64
Train years:
year
1990    166
2000    196
Name: count, dtype: int64

Test years:
year
2010    209
Name: count, dtype: int64
(362, 3) (209, 3)


,model,RMSE,MAE,R2_oos
2,Ridge,0.023648,0.016748,-0.037848
1,Linear,0.023672,0.016789,-0.039994
3,Lasso,0.024428,0.017832,-0.107477
0,Mean,0.025196,0.018678,-0.178204
